In [ ]:
%pip install -q datasets xgboost catboost scikit-learn pandas fastapi uvicorn pydantic nest-asyncio
print("All required libraries installed!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 7.9 MB/s eta 0:00:00
All required libraries installed!


In [ ]:
from datasets import load_dataset

dataset_stream = load_dataset("winterForestStump/10-K_sec_filings", split="001", streaming=True)

small_dataset = list(dataset_stream.take(50))

print(f"Successfully fetched {len(small_dataset)} documents instantly!")

first_doc = small_dataset[0]
print("\nKeys available in this dataset:", first_doc.keys())

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/7.35k [00:00<?, ?B/s]

Successfully fetched 50 documents instantly!

Keys available in this dataset: dict_keys(['cik', 'company_name', 'filing_date', 'Business', 'Risk Factors', 'Unresolved Staff Comments', 'Properties', 'Legal Proceedings', 'Mine Safety Disclosures', 'Market for Registrant’s Common Equity, Related Stockholder Matters and Issuer Purchases of Equity Securities', 'Selected Financial Data', 'Management’s Discussion and Analysis of Financial Condition and Results of Operations', 'Quantitative and Qualitative Disclosures about Market Risk', 'Financial Statements and Supplementary Data', 'Changes in and Disagreements with Accountants on Accounting and Financial Disclosure', 'Controls and Procedures', 'Other Information', 'Directors, Executive Officers and Corporate Governance', 'Executive Compensation', 'Security Ownership of Certain Beneficial Owners and Management and Related Stockholder Matters', 'Certain Relationships and Related Transactions, and Director Independence', 'Principal Accountant 

In [ ]:
from datasets import load_dataset, interleave_datasets
import pandas as pd
import re
from textblob import TextBlob
from sklearn.model_selection import train_test_split

stream1 = load_dataset("winterForestStump/10-K_sec_filings", split="001", streaming=True)
stream2 = load_dataset("winterForestStump/10-K_sec_filings", split="002", streaming=True)
combined_stream = interleave_datasets([stream1, stream2])
large_dataset = list(combined_stream.take(3000))
df = pd.DataFrame(large_dataset)

def clean_sec_text(raw_text):
    if not isinstance(raw_text, str):
        return ""
    text = raw_text.lower()
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

df['cleaned_text'] = df['Business'].apply(clean_sec_text)
def get_sentiment(text):
    polarity = TextBlob(str(text)).sentiment.polarity
    if polarity < -0.05: return 0
    elif polarity > 0.05: return 2
    else: return 1

df['label'] = df['cleaned_text'].apply(get_sentiment)
df = df[df['cleaned_text'].str.strip().astype(bool)]

X_train, X_test, y_train, y_test = train_test_split(
    df['cleaned_text'],
    df['label'],
    test_size=0.2,
    random_state=42,
    stratify=df['label']
)

print(f"Ready to proceed with {len(X_train)} training rows.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/7.35k [00:00<?, ?B/s]

Ready to proceed with 1712 training rows.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import joblib

print("Translating text into memory-safe TF-IDF arrays...")

vectorizer = TfidfVectorizer(max_features=1000, stop_words='english')

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

X_train_dense = X_train_tfidf.toarray()
X_test_dense = X_test_tfidf.toarray()

joblib.dump(vectorizer, "tfidf_vectorizer.joblib")

print("Vectorizer trained and saved successfully to 'tfidf_vectorizer.joblib'!")
print(f"Shape of training features: {X_train_dense.shape}")

Translating text into memory-safe TF-IDF arrays...
Vectorizer trained and saved successfully to 'tfidf_vectorizer.joblib'!
Shape of training features: (1712, 1000)


In [ ]:
from sklearn.ensemble import AdaBoostClassifier

print("Training AdaBoost...")
adaboost_model = AdaBoostClassifier(random_state=42)
adaboost_model.fit(X_train_dense, y_train)
print("AdaBoost trained successfully!")

Training AdaBoost...
AdaBoost trained successfully!


In [ ]:
from xgboost import XGBClassifier

print("Training XGBoost...")
xgboost_model = XGBClassifier(random_state=42, eval_metric='mlogloss', n_jobs=1)
xgboost_model.fit(X_train_dense, y_train)
print("XGBoost trained successfully!")

Training XGBoost...
XGBoost trained successfully!


In [ ]:
%pip install catboost

print("CatBoost has been successfully installed!")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 11.4 MB/s eta 0:00:00
CatBoost has been successfully installed!


In [ ]:
from catboost import CatBoostClassifier

print("Training CatBoost...")
catboost_model = CatBoostClassifier(random_state=42, verbose=0, thread_count=1)
catboost_model.fit(X_train_dense, y_train)
print("CatBoost trained successfully!")

Training CatBoost...
CatBoost trained successfully!


In [ ]:
trained_models = {
    "AdaBoost": adaboost_model,
    "XGBoost": xgboost_model,
    "CatBoost": catboost_model
}
print("All models are safely compiled in memory and ready for evaluation!")

All models are safely compiled in memory and ready for evaluation!


In [ ]:

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import pandas as pd
import numpy as np

results = []
best_model_name = None
best_f1 = 0

print("Evaluating Models...\n")

for name, model in trained_models.items():
    predictions = model.predict(X_test_dense)

    acc = accuracy_score(y_test, predictions)
    prec = precision_score(y_test, predictions, average='weighted', zero_division=0)
    rec = recall_score(y_test, predictions, average='weighted', zero_division=0)
    f1 = f1_score(y_test, predictions, average='weighted', zero_division=0)
    cm = confusion_matrix(y_test, predictions)

    if f1 > best_f1:
        best_f1 = f1
        best_model_name = name

    results.append({
        "Model": name,
        "Accuracy": np.round(acc, 4),
        "Precision": np.round(prec, 4),
        "Recall": np.round(rec, 4),
        "F1 Score": np.round(f1, 4)
    })

    print(f" {name} Confusion Matrix ")
    print(cm, "\n")

metrics_df = pd.DataFrame(results)
print(" FINAL PERFORMANCE COMPARISON")
print(metrics_df.to_string(index=False))
print(f"\n Best Performing Model: {best_model_name} (F1 Score: {np.round(best_f1, 4)})")

Evaluating Models...

 AdaBoost Confusion Matrix 
[[  0   2   0]
 [  0 112  48]
 [  0 104 163]] 

 XGBoost Confusion Matrix 
[[  0   1   1]
 [  0 114  46]
 [  0  28 239]] 

 CatBoost Confusion Matrix 
[[  0   1   1]
 [  0 108  52]
 [  0  28 239]] 

 FINAL PERFORMANCE COMPARISON
   Model  Accuracy  Precision  Recall  F1 Score
AdaBoost    0.6410     0.6724  0.6410    0.6455
 XGBoost    0.8228     0.8174  0.8228    0.8186
CatBoost    0.8089     0.8034  0.8089    0.8034

 Best Performing Model: XGBoost (F1 Score: 0.8186)


In [ ]:
from datasets import load_dataset, interleave_datasets
import pandas as pd
import re
from textblob import TextBlob
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from xgboost import XGBClassifier
import joblib

stream1 = load_dataset("winterForestStump/10-K_sec_filings", split="001", streaming=True)
stream2 = load_dataset("winterForestStump/10-K_sec_filings", split="002", streaming=True)
combined_stream = interleave_datasets([stream1, stream2])
large_dataset = list(combined_stream.take(3000))
df = pd.DataFrame(large_dataset)

def clean_sec_text(raw_text):
    if not isinstance(raw_text, str): return ""
    text = raw_text.lower()
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

df['cleaned_text'] = df['Business'].apply(clean_sec_text)

def get_sentiment(text):
    polarity = TextBlob(str(text)).sentiment.polarity
    if polarity < -0.05: return 0
    elif polarity > 0.05: return 2
    else: return 1

df['label'] = df['cleaned_text'].apply(get_sentiment)
df = df[df['cleaned_text'].str.strip().astype(bool)]

X_train, X_test, y_train, y_test = train_test_split(
    df['cleaned_text'], df['label'], test_size=0.2, random_state=42, stratify=df['label']
)

vectorizer = TfidfVectorizer(max_features=1000, stop_words='english')
X_train_dense = vectorizer.fit_transform(X_train).toarray()

xgboost_model = XGBClassifier(random_state=42, eval_metric='mlogloss', n_jobs=1)
xgboost_model.fit(X_train_dense, y_train)

joblib.dump(xgboost_model, "best_boosting_model.joblib")
joblib.dump(vectorizer, "tfidf_vectorizer.joblib")


['tfidf_vectorizer.joblib']

In [ ]:
import re
import joblib
import numpy as np
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI(title="SEC 10-K Document Classification API")

production_model = joblib.load("best_boosting_model.joblib")
production_vectorizer = joblib.load("tfidf_vectorizer.joblib")

class PredictRequest(BaseModel):
    text: str

def clean_sec_text(raw_text):
    if not isinstance(raw_text, str): return ""
    text = raw_text.lower()
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

@app.post("/predict")
def predict_document(payload: PredictRequest):
    cleaned = clean_sec_text(payload.text)
    features = production_vectorizer.transform([cleaned]).toarray()

    label = production_model.predict(features)[0]
    if hasattr(label, "__len__") and not isinstance(label, (str, int)):
        label = label[0]

    probabilities = production_model.predict_proba(features)[0]
    confidence = np.max(probabilities)

    return {
        "label": str(int(label)),
        "confidence": float(round(confidence, 4))
    }